# 01 · ETL — Peace Now: Assentaments i Outposts (1993–2024)

**Font:** Peace Now Settlement Watch  
**Fitxer font:** `data/raw/Settlements_and_Outposts_Peace_Now_07_2026.xlsx`  
**Data de publicació:** Juliol 2026  

**Outputs:**
- `data/clean/peacenow_population_long.csv` — sèrie temporal de població (format llarg)
- `data/clean/peacenow_settlements_geo.csv` — metadades geoespacials per assentament
- `data/clean/peacenow_outposts_geo.csv` — outposts geolocalitzats

---
## Rol en l'arquitectura del projecte

Peace Now és la **font principal** del pipeline de población i geometria.

| Output | Contingut | Usa al notebook |
|--------|-----------|-----------------|
| `peacenow_population_long.csv` | Sèrie temporal 1993–2024 | 03_merge_population |
| `peacenow_settlements_geo.csv` | Coordenades, tipologia, distància LV, fundació | 05_analysis (mapes) |
| `peacenow_outposts_geo.csv` | 383 outposts geolocalitzats amb any de fundació | 05_analysis (mapes) |

---
## Notes metodològiques

- **Rang temporal principal:** 1993–2024 (cobertura estable, 127–132 assentaments/any)
- **Dades anteriors a 1996:** disponibles des de 1969 però amb cobertura irregular; s'exclouen de la sèrie principal
- **Any 2024:** columna "Population May 2024" (dades de maig, no de final d'any). Es documenta a la metodologia
- **Gap 1984:** un sol any absent en 55 anys de sèrie; no afecta el rang 1993–2024
- **21 assentaments sense coordenades UTM:** tenen coordenades ITM; es convertiran si cal
- **Outposts:** no tenen dades de població, únicament coordenades i any de fundació


## 1. Importació de llibreries

In [2]:
import pandas as pd
import re
import os

print("Llibreries carregades correctament")

Llibreries carregades correctament


## 2. Configuració

In [5]:
FILE_PATH = "../Datasets/raw/Settlements_PeaceNow.xlsx"
SHEET_SETTLEMENTS  = "Settlements"
SHEET_OUTPOSTS     = "Outposts"
SHEET_LEGALIZATION = "Outpost Legalization"

# Rang temporal de la sèrie principal
YEAR_MIN = 1993
YEAR_MAX = 2024

OUTPUT_POP  = "data/clean/peacenow_population_long.csv"
OUTPUT_GEO  = "data/clean/peacenow_settlements_geo.csv"
OUTPUT_OUTPOSTS = "data/clean/peacenow_outposts_geo.csv"

print("Configuració carregada")

Configuració carregada


## 3. Lectura dels fulls

In [6]:
df_raw  = pd.read_excel(FILE_PATH, sheet_name=SHEET_SETTLEMENTS,  engine="openpyxl")
df_out  = pd.read_excel(FILE_PATH, sheet_name=SHEET_OUTPOSTS,     engine="openpyxl")
df_leg  = pd.read_excel(FILE_PATH, sheet_name=SHEET_LEGALIZATION, engine="openpyxl")  # reservat per a ús futur

print(f"Settlements:           {len(df_raw):3d} files | {len(df_raw.columns)} columnes")
print(f"Outposts:              {len(df_out):3d} files | {len(df_out.columns)} columnes")
print(f"Outpost Legalization:  {len(df_leg):3d} files | {len(df_leg.columns)} columnes")

Settlements:           148 files | 68 columnes
Outposts:              383 files | 13 columnes
Outpost Legalization:   68 files | 36 columnes


## 4. Neteja de la taula d'assentaments

Seleccionem i netegem les columnes de metadades geoespacials.  
Les columnes de població es tractaran a la secció 5.


In [ ]:
# Columnes de metadades (no-població)
META_COLS = {
    "Name":                                   "settlement",
    "Name Hebrew":                            "name_hebrew",
    "DB_ID":                                  "db_id",
    "x (UTM)":                                "x_utm",    # coordenada est (UTM)
    "y (UTM)":                                "y_utm",    # coordenada nord (UTM)
    "x (ITM)":                                "x_itm",
    "y (ITM)":                                "y_itm",
    "Year established":                       "year_established",
    "Distance from Green Line (Kilometers)":  "dist_green_line_km",
    "Municipality":                           "municipality",
    "Urban Pattern":                          "urban_pattern",
    "Elevation":                              "elevation_m",
    "Comments":                               "comments",
}

df_geo = df_raw[list(META_COLS.keys())].rename(columns=META_COLS).copy()

# Netejar tipus
df_geo["db_id"]            = pd.to_numeric(df_geo["db_id"], errors="coerce").astype("Int64")
df_geo["year_established"] = pd.to_numeric(df_geo["year_established"], errors="coerce").astype("Int64")
df_geo["dist_green_line_km"] = pd.to_numeric(df_geo["dist_green_line_km"], errors="coerce")
df_geo["elevation_m"]      = pd.to_numeric(df_geo["elevation_m"], errors="coerce")
df_geo["x_utm"]            = pd.to_numeric(df_geo["x_utm"], errors="coerce")
df_geo["y_utm"]            = pd.to_numeric(df_geo["y_utm"], errors="coerce")
df_geo["settlement"]       = df_geo["settlement"].str.strip()

print(f"Shape df_geo: {df_geo.shape}")
print(f"\nNuls per columna:")
print(df_geo.isnull().sum())

Shape df_geo: (148, 13)

Nuls per columna:
settlement              0
name_hebrew             0
db_id                   0
lat                     1
lon                     1
x_itm                   0
y_itm                   0
year_established        7
dist_green_line_km     11
municipality            0
urban_pattern          10
elevation_m            26
comments              120
dtype: int64


In [ ]:
# Distribució per tipologia i municipi
print("Urban Pattern:")
print(df_geo["urban_pattern"].value_counts())
print(f"\nMunicipis (consells regionals): {df_geo['municipality'].nunique()}")
print(df_geo["municipality"].value_counts())

Urban Pattern:
urban_pattern
Community           82
Urban               25
Moshav              15
Kibbutz             10
Cooperative          5
Moshav/Community     1
Name: count, dtype: int64

Municipis (consells regionals): 24
municipality
Binyamin          36
Shomron           30
Jordan Valley     24
Gush Etzion       16
Har Hebron        15
Megilot            4
Givat Ze'ev        3
Alfei Menashe      2
Beit Arye          2
Kiryat Arba        2
Elkana             1
Oranit             1
Beit El            1
Ariel              1
Beitar Illit       1
Efrata             1
Har Adar           1
Modi'in Illit      1
Ma'ale Adumim      1
Immanuel           1
Kedumim            1
Karnei Shomron     1
Shimon             1
Shaar Shomron      1
Name: count, dtype: int64


## 5. Extracció de la sèrie temporal de població

Identifiquem totes les columnes de població i les convertim a format llarg.  
Filtrem al rang 1993–2024 per a la sèrie principal.


In [ ]:
def extract_year_from_col(col_name):
    """Extreu l'any (int) d'una columna tipus 'Population 2023' o 'Population May 2024'."""
    match = re.search(r'(\d{4})', str(col_name))
    return int(match.group(1)) if match else None

# Identificar columnes de població dins del rang
pop_cols = []
for col in df_raw.columns:
    year = extract_year_from_col(col)
    if year and YEAR_MIN <= year <= YEAR_MAX:
        pop_cols.append((col, year))

print(f"Columnes de població al rang {YEAR_MIN}–{YEAR_MAX}: {len(pop_cols)}")
for col, year in pop_cols:
    print(f"  {col} → any {year}")

Columnes de població al rang 1993–2024: 32
  Population 1993 → any 1993
  Population 1994 → any 1994
  Population 1995 → any 1995
  Population 1996 → any 1996
  Population 1997 → any 1997
  Population 1998 → any 1998
  Population 1999 → any 1999
  Population 2000 → any 2000
  Population 2001 → any 2001
  Population 2002 → any 2002
  Population 2003 → any 2003
  Population 2004 → any 2004
  Population 2005 → any 2005
  Population 2006 → any 2006
  Population 2007 → any 2007
  Population 2008 → any 2008
  Population 2009 → any 2009
  Population 2010 → any 2010
  Population 2011 → any 2011
  Population 2012 → any 2012
  Population 2013 → any 2013
  Population 2014 → any 2014
  Population 2015 → any 2015
  Population 2016 → any 2016
  Population 2017 → any 2017
  Population 2018 → any 2018
  Population 2019 → any 2019
  Population 2020 → any 2020
  Population 2021 → any 2021
  Population 2022 → any 2022
  Population 2023 → any 2023
  Population May 2024 → any 2024


In [ ]:
# Construir DataFrame ample amb les columnes de població seleccionades
pop_col_names = [col for col, _ in pop_cols]
year_map      = {col: year for col, year in pop_cols}

df_pop_wide = df_raw[["Name"] + pop_col_names].copy()
df_pop_wide = df_pop_wide.rename(columns={"Name": "settlement"})
df_pop_wide["settlement"] = df_pop_wide["settlement"].str.strip()

# Convertir a format llarg (melt)
df_pop_long = df_pop_wide.melt(
    id_vars=["settlement"],
    value_vars=pop_col_names,
    var_name="col_raw",
    value_name="population"
)

# Afegir any
df_pop_long["year"] = df_pop_long["col_raw"].map(year_map)

# Eliminar files sense dada de població
df_pop_long["population"] = pd.to_numeric(df_pop_long["population"], errors="coerce")
df_pop_long = df_pop_long.dropna(subset=["population"])
df_pop_long["population"] = df_pop_long["population"].round().astype(int)
df_pop_long["year"]       = df_pop_long["year"].astype(int)

# Eliminar assentaments amb població 0 (fundats però no habitats aquell any)
df_pop_long = df_pop_long[df_pop_long["population"] > 0]

# Afegir font
df_pop_long["source"] = "peace_now"

# Columnes finals
df_pop_long = df_pop_long[["settlement", "year", "population", "source"]]
df_pop_long = df_pop_long.sort_values(["settlement", "year"]).reset_index(drop=True)

print(f"Shape format llarg: {df_pop_long.shape}")
print(f"Anys: {df_pop_long['year'].min()}–{df_pop_long['year'].max()}")
print(f"Assentaments únics: {df_pop_long['settlement'].nunique()}")

Shape format llarg: (3861, 4)
Anys: 1993–2024
Assentaments únics: 132


## 6. Comprovacions de qualitat

In [ ]:
# Cobertura per any
cobertura = df_pop_long.groupby("year").agg(
    n_settlements=("settlement", "count"),
    pop_total=("population", "sum")
).reset_index()

print("Cobertura per any:")
print(cobertura.to_string(index=False))

Cobertura per any:
 year  n_settlements  pop_total
 1993            105     110066
 1994            110     124005
 1995            110     137466
 1996            116     139453
 1997            116     152277
 1998            118     165540
 1999            118     174405
 2000            118     191125
 2001            118     201674
 2002            118     212218
 2003            119     221898
 2004            118     233471
 2005            119     247230
 2006            119     261537
 2007            119     276045
 2008            119     290311
 2009            121     297386
 2010            120     310990
 2011            121     325452
 2012            123     341418
 2013            125     355983
 2014            125     369977
 2015            126     385264
 2016            126     398534
 2017            126     412668
 2018            127     427072
 2019            127     440820
 2020            126     450711
 2021            127     464458
 2022            127 

In [ ]:
# Duplicats
dups = df_pop_long.duplicated(subset=["settlement", "year"])
print(f"Duplicats (settlement + year): {dups.sum()}")

# Nuls
print(f"Nuls per columna:")
print(df_pop_long.isnull().sum())

# Exemple: assentaments coneguts
for name in ["Ariel", "Beitar Illit", "Modi'in Illit", "Ma'ale Adumim"]:
    row = df_pop_long[(df_pop_long["settlement"] == name) & 
                      (df_pop_long["year"] == 2023)]
    if len(row) > 0:
        print(f"  {name} (2023): {row['population'].values[0]:,}")
    else:
        print(f"  {name} (2023): NOT FOUND")

Duplicats (settlement + year): 0
Nuls per columna:
settlement    0
year          0
population    0
source        0
dtype: int64
  Ariel (2023): 21,841
  Beitar Illit (2023): 69,281
  Modi'in Illit (2023): NOT FOUND
  Ma'ale Adumim (2023): 36,680


In [ ]:
# Nota metodològica — any 2024:
# La columna original és 'Population May 2024' (dades de maig, no de final d'any).
# Es tracta com a any 2024 a la sèrie. Cal documentar-ho a l'informe final.
# Referència CBS desembre 2023: 503.732 colons
# Referència CBS desembre 2024: ~517.000 (estimació)
print(f"Total documentat (maig 2024): {df_pop_long[df_pop_long['year']==2024]['population'].sum():,.0f}")

Nota metodològica — any 2024:
La columna original és 'Population May 2024' (dades de maig, no de final d'any).
Es tracta com a any 2024 a la sèrie. Cal documentar-ho a l'informe final.
Total documentat (maig 2024): 496,852
Referència CBS desembre 2023: 503,732
Referència CBS desembre 2024: ~517,000 (estimació)


## 7. Neteja dels outposts

Els outposts no tenen dades de població però estan tots geolocalitzats.  
Són especialment rellevants per analitzar l'acceleració post-7O (2023–2026).


In [ ]:
OUTPOST_COLS = {
    "Name":             "name",
    "Name Hebrew":      "name_hebrew",
    "DB_ID":            "db_id",
    "Settlement Type":  "settlement_type",
    "Established Year": "year_established",
    "x (UTM)":          "x_utm",
    "y(UTM)":           "y_utm",
    "x (ITM)":          "x_itm",
    "y (ITM)":          "y_itm",
    "Nearest Settlemet":"nearest_settlement",
    "District":         "district",
    "Municipality":     "municipality",
}

df_outposts = df_out[list(OUTPOST_COLS.keys())].rename(columns=OUTPOST_COLS).copy()

df_outposts["db_id"]            = pd.to_numeric(df_outposts["db_id"], errors="coerce").astype("Int64")
df_outposts["year_established"] = pd.to_numeric(df_outposts["year_established"], errors="coerce").astype("Int64")
df_outposts["x_utm"]            = pd.to_numeric(df_outposts["x_utm"], errors="coerce")
df_outposts["y_utm"]            = pd.to_numeric(df_outposts["y_utm"], errors="coerce")
df_outposts["name"]             = df_outposts["name"].str.strip()

print(f"Shape outposts: {df_outposts.shape}")
print(f"Outposts amb coordenades: {df_outposts['lat'].notna().sum()}")
print(f"\nOutposts per any de fundació (des de 2020):")
recent = df_outposts[df_outposts["year_established"] >= 2020]
print(recent["year_established"].value_counts().sort_index())
print(f"\nOutposts des del 7-O (oct 2023–2026): {len(df_outposts[df_outposts['year_established'] >= 2023])}")

Shape outposts: (383, 12)
Outposts amb coordenades: 383

Outposts per any de fundació (des de 2020):
year_established
2020    11
2021    14
2022     5
2023    32
2024    61
2025    91
2026    48
Name: count, dtype: Int64

Outposts des del 7-O (oct 2023–2026): 232


## 8. Exportació

In [ ]:
os.makedirs("data/clean", exist_ok=True)

# 1. Sèrie temporal de població
df_pop_long.to_csv(OUTPUT_POP, index=False)
print(f"✓ {OUTPUT_POP}")
print(f"  {df_pop_long.shape[0]:,} registres | {df_pop_long['settlement'].nunique()} assentaments | "
      f"anys {df_pop_long['year'].min()}–{df_pop_long['year'].max()}")

# 2. Metadades geoespacials
df_geo.to_csv(OUTPUT_GEO, index=False)
print(f"\n✓ {OUTPUT_GEO}")
print(f"  {len(df_geo)} assentaments | columnes: {list(df_geo.columns)}")

# 3. Outposts
df_outposts.to_csv(OUTPUT_OUTPOSTS, index=False)
print(f"\n✓ {OUTPUT_OUTPOSTS}")
print(f"  {len(df_outposts)} outposts | columnes: {list(df_outposts.columns)}")

✓ data/clean/peacenow_population_long.csv
  3,861 registres | 132 assentaments | anys 1993–2024

✓ data/clean/peacenow_settlements_geo.csv
  148 assentaments | columnes: ['settlement', 'name_hebrew', 'db_id', 'lat', 'lon', 'x_itm', 'y_itm', 'year_established', 'dist_green_line_km', 'municipality', 'urban_pattern', 'elevation_m', 'comments']

✓ data/clean/peacenow_outposts_geo.csv
  383 outposts | columnes: ['name', 'name_hebrew', 'db_id', 'settlement_type', 'year_established', 'lat', 'lon', 'x_itm', 'y_itm', 'nearest_settlement', 'district', 'municipality']
